fffffffffffffffff

In [ ]:
# 1. Install Dependencies
!pip install -q -U transformers accelerate langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu pandas scikit-learn

import time
import json
import re
import string
from collections import Counter
import pandas as pd
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# 2. Load Dataset
try:
    with open('cleaned_tatqa_rag_data.json', 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    eval_dataset = dataset[:30] # Subset for rapid evaluation
    print(f"Loaded dataset. Evaluating on {len(eval_dataset)} questions.")
except FileNotFoundError:
    print("Dataset not found. Please ensure 'cleaned_tatqa_rag_data.json' is in your workspace.")
    dataset, eval_dataset = [], []

# 3. Define Evaluation Metrics
def normalize_answer(s):
    s = str(s).lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    return ' '.join(s.split())

def compute_em(prediction, ground_truth):
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))

def compute_f1(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()
    if not pred_tokens or not truth_tokens:
        return int(pred_tokens == truth_tokens)
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())
    if num_same == 0: return 0
    precision = 1.0 * num_same / len(pred_tokens)
    recall = 1.0 * num_same / len(truth_tokens)
    return (2 * precision * recall) / (precision + recall)

def extract_numbers(text):
    return re.findall(r"[-+]?(?:\d*\.\d+|\d+)", str(text))

def compute_numerical_accuracy(prediction, ground_truth):
    pred_nums = extract_numbers(normalize_answer(prediction))
    truth_nums = extract_numbers(normalize_answer(ground_truth))
    if not truth_nums: return None
    return int(any(t in pred_nums for t in truth_nums))

def compute_retrieval_metrics(retrieved_ids, ground_truth_id, k=3):
    top_k_ids = retrieved_ids[:k]
    hit = 1 if ground_truth_id in top_k_ids else 0
    return hit, hit / k

# 4. Load LLM Locally into Colab T4 GPU (Bypasses ALL rate limits and APIs)
print("\nLoading Qwen-2.5-3B into GPU VRAM (This takes ~2 minutes)...")
model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=64, # Kept short to boost Exact Match scoring
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False
)

def ask_llm(system_prompt, user_prompt):
    """Formats prompts using the model's native chat template for accurate generation."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    outputs = pipe(prompt)
    return outputs[0]['generated_text'].strip()

# 5. LLM Baseline Evaluation
baseline_results = []
total_baseline_latency = 0

print("Running LLM Baseline Evaluation (No RAG)...")
system_instruction = "You are a financial AI assistant. Answer the user's question directly. Provide ONLY the final numerical value, phrase, or entity. Do not write full sentences."

for item in eval_dataset:
    question = item['question']
    ground_truth = item['ground_truth']['answer']

    start_time = time.time()
    prediction = ask_llm(system_instruction, question)
    latency = time.time() - start_time

    total_baseline_latency += latency
    baseline_results.append({
        'qid': item['id'],
        'answer_type': item['metadata']['answer_type'],
        'em': compute_em(prediction, ground_truth),
        'f1': compute_f1(prediction, ground_truth),
        'num_acc': compute_numerical_accuracy(prediction, ground_truth),
        'latency': latency
    })

df_baseline = pd.DataFrame(baseline_results)

# 6. Optimized RAG Pipeline Evaluation
print("\nBuilding Focused FAISS Vector Store...")
documents = []

for item in eval_dataset:
    text_evidence = item['evidence']['text']
    table_evidence = " | ".join([str(row) for row in item['evidence']['table']])
    doc = Document(page_content=f"Text: {text_evidence}\nTable: {table_evidence}", metadata={"source_id": item['id']})
    documents.append(doc)

# Runs embeddings on the GPU for maximum speed
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={'device': 'cuda'})
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

rag_results = []
total_rag_latency = 0

print("Running Basic RAG Evaluation...")
for item in eval_dataset:
    question = item['question']
    ground_truth = item['ground_truth']['answer']

    start_time = time.time()

    # Retrieval
    retrieved_docs = retriever.invoke(question)
    retrieved_ids = [doc.metadata['source_id'] for doc in retrieved_docs]
    retrieved_context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    recall_at_3, precision_at_3 = compute_retrieval_metrics(retrieved_ids, item['id'], k=3)

    # Augmented Generation
    user_prompt = f"Use the following financial evidence to answer the question. Output ONLY the exact value.\n\nEvidence:\n{retrieved_context}\n\nQuestion: {question}"
    prediction = ask_llm(system_instruction, user_prompt)
    latency = time.time() - start_time

    total_rag_latency += latency
    rag_results.append({
        'qid': item['id'],
        'answer_type': item['metadata']['answer_type'],
        'em': compute_em(prediction, ground_truth),
        'f1': compute_f1(prediction, ground_truth),
        'num_acc': compute_numerical_accuracy(prediction, ground_truth),
        'recall@3': recall_at_3,
        'precision@3': precision_at_3,
        'latency': latency
    })

df_rag = pd.DataFrame(rag_results)

# 7. Final System Comparison
comparison_df = pd.DataFrame({
    'Metric': ['Exact Match (EM)', 'F1 Score', 'Numerical Accuracy', 'Latency per Query (s)', 'Total Cost ($)', 'Recall@3', 'Precision@3'],
    'LLM Baseline': [
        df_baseline['em'].mean(), df_baseline['f1'].mean(), df_baseline['num_acc'].dropna().mean(),
        df_baseline['latency'].mean(), 0.00, "N/A", "N/A"
    ],
    'Basic RAG': [
        df_rag['em'].mean(), df_rag['f1'].mean(), df_rag['num_acc'].dropna().mean(),
        df_rag['latency'].mean(), 0.00, df_rag['recall@3'].mean(), df_rag['precision@3'].mean()
    ]
})

print("\n=== SYSTEM COMPARISON: LLM-Only vs Basic RAG ===")
print(comparison_df.to_string(index=False))

print("\n--- Baseline Answer-Type Accuracy (EM) ---")
print(df_baseline.groupby('answer_type')['em'].mean())

print("\n--- RAG Answer-Type Accuracy (EM) ---")
print(df_rag.groupby('answer_type')['em'].mean())

Loaded dataset. Evaluating on 30 questions.

Loading Qwen-2.5-3B into GPU VRAM (This takes ~2 minutes)...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Running LLM Baseline Evaluation (No RAG)...


[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length


Building Focused FAISS Vector Store...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running Basic RAG Evaluation...


[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs


=== SYSTEM COMPARISON: LLM-Only vs Basic RAG ===
               Metric LLM Baseline  Basic RAG
     Exact Match (EM)          0.0   0.033333
             F1 Score     0.035341   0.050000
   Numerical Accuracy         0.05   0.100000
Latency per Query (s)     0.650736   0.538711
       Total Cost ($)          0.0   0.000000
             Recall@3          N/A   0.100000
          Precision@3          N/A   0.033333

--- Baseline Answer-Type Accuracy (EM) ---
answer_type
arithmetic    0.0
multi-span    0.0
span          0.0
Name: em, dtype: float64

--- RAG Answer-Type Accuracy (EM) ---
answer_type
arithmetic    0.0
multi-span    0.0
span          0.1
Name: em, dtype: float64


In [ ]:
# Cell 1: Install Dependencies
!pip install -q -U rank-bm25 sentence-transformers transformers accelerate faiss-cpu pandas scikit-learn

In [ ]:
# Cell 2: Hybrid RAG Implementation and Evaluation
import re
import json
import math
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

# 1. Load Dataset
with open('cleaned_tatqa_rag_data.json', 'r', encoding='utf-8') as f:
    corpus_dataset = json.load(f)

eval_subset = corpus_dataset[:30]

# Build corpus documents
corpus_docs = []
corpus_ids = []
for item in corpus_dataset:
    text_ev = item['evidence']['text']
    table_ev = " | ".join([str(row) for row in item['evidence']['table']])
    content = f"Text: {text_ev}\nTable: {table_ev}"
    corpus_docs.append(content)
    corpus_ids.append(item['id'])

id_to_idx = {cid: idx for idx, cid in enumerate(corpus_ids)}

# 2. Initialize Models
print("Initializing Dense Embedder and Cross-Encoder Reranker...")
dense_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cuda')

# Compute Dense Embeddings
doc_embeddings = dense_model.encode(corpus_docs, convert_to_tensor=True, show_progress_bar=False)

# Initialize BM25 Index
tokenized_corpus = [doc.lower().split() for doc in corpus_docs]
bm25 = BM25Okapi(tokenized_corpus)

# 3. Retrieval Pipelines
def retrieve_dense(query, top_k=10):
    q_emb = dense_model.encode(query, convert_to_tensor=True)
    cos_scores = (doc_embeddings @ q_emb).cpu().numpy()
    top_indices = np.argsort(-cos_scores)[:top_k]
    return [corpus_ids[i] for i in top_indices]

def retrieve_bm25(query, top_k=10):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(-scores)[:top_k]
    return [corpus_ids[i] for i in top_indices]

def retrieve_hybrid_rrf_rerank(query, top_k_fusion=20, final_k=3, rrf_k=60):
    # Step A: Dense + BM25 Candidates
    dense_candidates = retrieve_dense(query, top_k=top_k_fusion)
    bm25_candidates = retrieve_bm25(query, top_k=top_k_fusion)

    # Step B: Reciprocal Rank Fusion
    rrf_scores = {}
    for rank, doc_id in enumerate(dense_candidates):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (rrf_k + rank + 1))
    for rank, doc_id in enumerate(bm25_candidates):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (rrf_k + rank + 1))

    fused_candidates = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:top_k_fusion]

    # Step C: Cross-Encoder Reranking
    pairs = [[query, corpus_docs[id_to_idx[doc_id]]] for doc_id in fused_candidates]
    rerank_scores = reranker_model.predict(pairs)
    ranked_indices = np.argsort(-rerank_scores)[:final_k]

    return [fused_candidates[i] for i in ranked_indices]

# 4. Evaluation Engine (Recall@K, Precision@K, MRR, NDCG, Wrong-Evidence Rate)
def compute_ndcg_at_k(retrieved, target_id, k=3):
    for i, doc_id in enumerate(retrieved[:k]):
        if doc_id == target_id:
            return 1.0 / math.log2(i + 2)
    return 0.0

def evaluate_retrievers(dataset, k=3):
    results = {'Dense-Only': [], 'BM25-Only': [], 'Hybrid+Rerank': []}

    financial_terms = re.compile(r'\b(revenue|margin|ebitda|income|cash|growth|cost|shares|operating)\b', re.I)
    year_pattern = re.compile(r'\b(19\d\d|20\d\d)\b')

    for item in dataset:
        qid = item['id']
        query = item['question']
        target_id = qid

        has_year = bool(year_pattern.search(query))
        has_fin_term = bool(financial_terms.search(query))

        runs = {
            'Dense-Only': retrieve_dense(query, top_k=k),
            'BM25-Only': retrieve_bm25(query, top_k=k),
            'Hybrid+Rerank': retrieve_hybrid_rrf_rerank(query, final_k=k)
        }

        for name, retrieved in runs.items():
            hit = 1 if target_id in retrieved[:k] else 0
            rank = retrieved.index(target_id) + 1 if hit else 0
            mrr = (1.0 / rank) if hit else 0.0
            ndcg = compute_ndcg_at_k(retrieved, target_id, k=k)
            wrong_evidence = 1 if (retrieved and retrieved[0] != target_id) else 0

            results[name].append({
                'hit': hit,
                'precision': hit / k,
                'mrr': mrr,
                'ndcg': ndcg,
                'wrong_evidence': wrong_evidence,
                'has_year': has_year,
                'has_fin_term': has_fin_term
            })

    summary = []
    for name in results:
        df = pd.DataFrame(results[name])
        summary.append({
            'Pipeline': name,
            f'Recall@{k}': df['hit'].mean(),
            f'Precision@{k}': df['precision'].mean(),
            'MRR': df['mrr'].mean(),
            f'NDCG@{k}': df['ndcg'].mean(),
            'Wrong-Evidence Rate': df['wrong_evidence'].mean(),
            'Recall (Exact Years)': df[df['has_year']]['hit'].mean(),
            'Recall (Fin Terms)': df[df['has_fin_term']]['hit'].mean()
        })
    return pd.DataFrame(summary)

df_retrieval_comparison = evaluate_retrievers(eval_subset, k=3)
print("\n=== STEP 1: RETRIEVAL SYSTEM EVALUATION ===")
print(df_retrieval_comparison.to_string(index=False))

Initializing Dense Embedder and Cross-Encoder Reranker...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


=== STEP 1: RETRIEVAL SYSTEM EVALUATION ===
     Pipeline  Recall@3  Precision@3  MRR  NDCG@3  Wrong-Evidence Rate  Recall (Exact Years)  Recall (Fin Terms)
   Dense-Only       0.0          0.0  0.0     0.0                  1.0                   0.0                 0.0
    BM25-Only       0.0          0.0  0.0     0.0                  1.0                   0.0                 0.0
Hybrid+Rerank       0.0          0.0  0.0     0.0                  1.0                   0.0                 0.0


In [ ]:
# Cell 3: Financial Calculation Engine and Evaluation
import re
import time
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Load Local LLM
model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

def run_llm(prompt, max_tokens=128):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

# 2. Sandboxed Deterministic Python Executor
def execute_python_code(code_str):
    """Executes arithmetic expression safely, blocking unsafe imports/globals."""
    safe_env = {
        "__builtins__": {},
        "abs": abs, "round": round, "min": min, "max": max, "sum": sum, "pow": pow
    }
    # Clean markdown and isolate executable lines
    clean_code = re.sub(r"```(?:python)?|```", "", code_str).strip()
    match = re.search(r"result\s*=\s*(.+)", clean_code)
    expression = match.group(1) if match else clean_code

    # Sanitize tokens to allow only valid mathematical operations
    if any(forbidden in expression for forbidden in ['import', 'os', 'sys', '__', 'eval', 'exec', 'open']):
        return None, "Security Violation"

    try:
        val = eval(expression, safe_env)
        return float(val), "Success"
    except Exception as e:
        return None, str(e)

# 3. Normalization and Numerical Parsing
def extract_first_number(s):
    nums = re.findall(r"[-+]?(?:\d*\.\d+|\d+)", str(s).replace(',', ''))
    return float(nums[0]) if nums else None

def is_numerically_close(pred, truth, tolerance=1e-2):
    if pred is None or truth is None:
        return False
    return math.isclose(pred, truth, rel_tol=tolerance, abs_tol=tolerance)

# 4. Evaluation: LLM-Only vs. Calculation Tools
calc_subset = [item for item in corpus_dataset if item['ground_truth'].get('derivation')][:30]

llm_only_results = []
tool_results = []

print("Running Financial Calculation Tools Benchmark...")

for item in calc_subset:
    q = item['question']
    gt_answer = item['ground_truth']['answer']
    gt_val = extract_first_number(gt_answer)
    context = f"Text: {item['evidence']['text']}\nTable: {' | '.join([str(r) for r in item['evidence']['table']])}"

    # --- Mode A: LLM-Only (Mental Arithmetic) ---
    prompt_llm = f"<|im_start|>system\nYou are a financial AI. Provide ONLY the final numerical answer.<|im_end|>\n<|im_start|>user\nContext:\n{context}\n\nQuestion: {q}<|im_end|>\n<|im_start|>assistant\n"
    raw_pred = run_llm(prompt_llm, max_tokens=32)
    llm_num = extract_first_number(raw_pred)

    llm_correct = is_numerically_close(llm_num, gt_val)
    llm_only_results.append({
        'qid': item['id'],
        'exact_match': int(llm_correct),
        'arithmetic_error': int(not llm_correct) if llm_num is not None else 1
    })

    # --- Mode B: Tool-Assisted (LLM identifies op -> Python executes) ---
    prompt_tool = (
        f"<|im_start|>system\nYou are a financial code engine. Formulate ONLY the direct Python arithmetic expression assigned to 'result'. "
        f"Do not write explanations. Example: result = (450 - 300) / 300 * 100<|im_end|>\n"
        f"<|im_start|>user\nContext:\n{context}\n\nQuestion: {q}<|im_end|>\n<|im_start|>assistant\nresult = "
    )
    code_generated = "result = " + run_llm(prompt_tool, max_tokens=48)
    computed_val, status = execute_python_code(code_generated)

    tool_success = int(status == "Success")
    tool_correct = int(is_numerically_close(computed_val, gt_val))

    tool_results.append({
        'qid': item['id'],
        'calc_success': tool_success,
        'exact_match': tool_correct,
        'arithmetic_error': int(not tool_correct) if tool_success else 1
    })

df_llm_only = pd.DataFrame(llm_only_results)
df_tool = pd.DataFrame(tool_results)

df_step2_comparison = pd.DataFrame({
    'Metric': [
        'Numerical Exact-Match Accuracy',
        'Arithmetic Error Rate',
        'Calculation Success Rate'
    ],
    'LLM-Only (Mental Math)': [
        df_llm_only['exact_match'].mean(),
        df_llm_only['arithmetic_error'].mean(),
        1.0 # Standard generation always completes
    ],
    'Tool-Assisted (Python Sandbox)': [
        df_tool['exact_match'].mean(),
        df_tool['arithmetic_error'].mean(),
        df_tool['calc_success'].mean()
    ]
})

print("\n=== STEP 2: FINANCIAL CALCULATION TOOL EVALUATION ===")
print(df_step2_comparison.to_string(index=False))

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Running Financial Calculation Tools Benchmark...

=== STEP 2: FINANCIAL CALCULATION TOOL EVALUATION ===
                        Metric  LLM-Only (Mental Math)  Tool-Assisted (Python Sandbox)
Numerical Exact-Match Accuracy                0.133333                        0.033333
         Arithmetic Error Rate                0.866667                        0.966667
      Calculation Success Rate                1.000000                        0.933333


In [ ]:
import re
import pandas as pd

def normalize_temporal_token(token):
    token = token.lower().strip()
    # Normalize FY patterns: 'fy19', 'fy 2019', 'fiscal year 2019' -> '2019'
    fy_match = re.search(r'(?:fy\s*|fiscal\s*(?:year)?\s*)?(\d{2,4})', token)
    if fy_match:
        val = fy_match.group(1)
        return f"20{val}" if len(val) == 2 else val
    return token

def extract_temporal_periods_canonical(text):
    raw_periods = re.findall(
        r'\b(?:19\d\d|20\d\d|Q[1-4]|first quarter|second quarter|third quarter|fourth quarter|\bFY\s*\d{2,4})\b',
        text,
        re.I
    )
    return {normalize_temporal_token(p) for p in raw_periods}

def extract_financial_entities(text):
    # Matches common capitalized corporate entities, reporting segments, or metrics
    entities = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', text)
    stopwords = {"The", "This", "In", "Table", "Text", "According", "For"}
    return {e for e in entities if e not in stopwords}

# Improved Evaluation Pipeline
verification_records = []

for item in eval_subset:
    qid = item['id']
    target_evidence = f"Text: {item['evidence']['text']}\nTable: {' | '.join([str(r) for r in item['evidence']['table']])}"
    gt_periods = extract_temporal_periods_canonical(target_evidence)
    gt_entities = extract_financial_entities(target_evidence)

    prompt_answer = (
        f"<|im_start|>system\nYou are a financial analyst. Answer the question based on context. "
        f"Include citations like [Doc 1] after claims.<|im_end|>\n"
        f"<|im_start|>user\nContext [Doc 1]: {target_evidence}\n\nQuestion: {item['question']}<|im_end|>\n<|im_start|>assistant\n"
    )
    generated_answer = run_llm(prompt_answer, max_tokens=128)

    claims = decompose_into_claims(generated_answer)
    total_citations = len(re.findall(r'\[Doc \d+\]', generated_answer))
    valid_citations = 0
    entailed_claims = 0

    period_hits, total_periods = 0, 0
    entity_hits, total_entities = 0, 0

    for claim in claims:
        # Entailment
        verdict = verify_claim_entailment(claim, target_evidence)
        if verdict == "SUPPORTED":
            entailed_claims += 1
            if '[Doc' in claim:
                valid_citations += 1

        # Canonical Period Matching
        claim_periods = extract_temporal_periods_canonical(claim)
        for p in claim_periods:
            total_periods += 1
            if p in gt_periods:
                period_hits += 1

        # Entity Matching
        claim_entities = extract_financial_entities(claim)
        for ent in claim_entities:
            total_entities += 1
            if any(ent.lower() in gte.lower() for gte in gt_entities):
                entity_hits += 1

    num_claims = max(len(claims), 1)

    citation_precision = (valid_citations / total_citations) if total_citations > 0 else 1.0
    citation_recall = 1.0 if entailed_claims > 0 and total_citations > 0 else 0.0
    entailment_acc = entailed_claims / num_claims
    unsupported_rate = (num_claims - entailed_claims) / num_claims

    p_score = (period_hits / total_periods) if total_periods > 0 else 1.0
    e_score = (entity_hits / total_entities) if total_entities > 0 else 1.0
    joint_entity_period_acc = (p_score + e_score) / 2.0

    verification_records.append({
        'qid': qid,
        'citation_precision': citation_precision,
        'citation_recall': citation_recall,
        'entailment_acc': entailment_acc,
        'unsupported_rate': unsupported_rate,
        'period_accuracy': p_score,
        'entity_period_accuracy': joint_entity_period_acc
    })

df_verify = pd.DataFrame(verification_records)

df_step3_summary = pd.DataFrame([{
    'Citation Precision': df_verify['citation_precision'].mean(),
    'Citation Recall': df_verify['citation_recall'].mean(),
    'Claim Entailment Accuracy': df_verify['entailment_acc'].mean(),
    'Unsupported-Claim Rate': df_verify['unsupported_rate'].mean(),
    'Entity & Period Accuracy': df_verify['entity_period_accuracy'].mean()
}])

print("\n=== UPDATED STEP 3: CLAIM-LEVEL EVIDENCE VERIFICATION EVALUATION ===")
print(df_step3_summary.to_string(index=False))


=== UPDATED STEP 3: CLAIM-LEVEL EVIDENCE VERIFICATION EVALUATION ===
 Citation Precision  Citation Recall  Claim Entailment Accuracy  Unsupported-Claim Rate  Entity & Period Accuracy
                1.0         0.933333                        1.0                     0.0                  0.272639
